# Qwen3-14B Fine-tuning — MAP: Charting Student Math Misunderstandings

**Strategy:** 4-bit QLoRA fine-tuning on Qwen3-14B for sequence classification.  
**Modes:** Set `QUICK_RUN = True` to verify the pipeline end-to-end, then `False` for full training.

## 1. Install Dependencies

In [10]:
!pip install -q peft bitsandbytes transformers datasets accelerate

## 2. Configuration

In [11]:
import os
from pathlib import Path

# ── Run mode ──────────────────────────────────────────────────────────────────
QUICK_RUN = True   # True → smoke-test on small subset | False → full training

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_PATH  = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/train.csv"
TEST_PATH   = "/kaggle/input/datasets/mkhanshovon43/map-ds/DS/test.csv"
OUTPUT_ROOT = Path("/kaggle/working/map_multi_llm_runs")

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME  = "Qwen/Qwen3-14B"          # used for loading AND output folder name
MODEL_SLUG  = MODEL_NAME.split("/")[-1] # → "Qwen3-14B"  (used in save path)

# ── HuggingFace token (from Kaggle Secrets) ───────────────────────────────────
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_API_KEY")

# ── Training hyperparameters ──────────────────────────────────────────────────
QUICK_SAMPLES   = 256           # samples used in quick_run mode
MAX_LENGTH      = 128 if QUICK_RUN else 256
EPOCHS          = 1   if QUICK_RUN else 3
BATCH_SIZE      = 4   if QUICK_RUN else 8
GRAD_ACCUM      = 2   if QUICK_RUN else 4
LEARNING_RATE   = 2e-4
WARMUP_RATIO    = 0.1

# ── LoRA hyperparameters ──────────────────────────────────────────────────────
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.1
LORA_TARGETS    = ["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ── Output dirs ───────────────────────────────────────────────────────────────
SAVE_DIR   = OUTPUT_ROOT / MODEL_SLUG / "best"   # final model checkpoint
TRAIN_DIR  = OUTPUT_ROOT / MODEL_SLUG / "train_output"
LOG_DIR    = OUTPUT_ROOT / MODEL_SLUG / "logs"
# CACHE_DIR  = Path("/kaggle/working/hf_cache")
CACHE_DIR = Path("/kaggle/tmp/hf_cache")

for d in [SAVE_DIR, TRAIN_DIR, LOG_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"{'[QUICK RUN]' if QUICK_RUN else '[FULL TRAINING]'} — model: {MODEL_NAME}")
print(f"  epochs={EPOCHS} | batch={BATCH_SIZE} | grad_accum={GRAD_ACCUM} | max_len={MAX_LENGTH}")
print(f"  save → {SAVE_DIR}")

[QUICK RUN] — model: Qwen/Qwen3-14B
  epochs=1 | batch=4 | grad_accum=2 | max_len=128
  save → /kaggle/working/map_multi_llm_runs/Qwen3-14B/best


## 3. Imports

In [12]:
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

PyTorch  : 2.9.0+cu126
CUDA     : True — NVIDIA H100 80GB HBM3


## 4. Data Loading & Preprocessing

In [13]:
train = pd.read_csv(TRAIN_PATH)
print(f"Raw train shape: {train.shape}")
train.head(3)

Raw train shape: (36696, 7)


,row_id,QuestionId,QuestionText,MC_Answer,StudentExplanation,Category,Misconception
0,0,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct,NaN
1,1,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third...,True_Correct,NaN
2,2,31772,What fraction of the shape is not shaded? Give...,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to un...",True_Neither,NaN


In [14]:
# ── Target labels ─────────────────────────────────────────────────────────────
train["Misconception"] = train["Misconception"].fillna("NA")
train["target"]        = train["Category"] + ":" + train["Misconception"]

le = LabelEncoder()
train["label"] = le.fit_transform(train["target"])
n_classes      = len(le.classes_)
print(f"Target classes : {n_classes}")

# ── is_correct flag ───────────────────────────────────────────────────────────
idx     = train.apply(lambda r: r.Category.split("_")[0], axis=1) == "True"
correct = train.loc[idx].copy()
correct["c"] = correct.groupby(["QuestionId", "MC_Answer"]).MC_Answer.transform("count")
correct = (correct
           .sort_values("c", ascending=False)
           .drop_duplicates("QuestionId")[["QuestionId", "MC_Answer"]])
correct["is_correct"] = 1

train = train.merge(correct, on=["QuestionId", "MC_Answer"], how="left")
train["is_correct"] = train["is_correct"].fillna(0).astype(int)

# ── Prompt formatting ─────────────────────────────────────────────────────────
def format_input(row):
    return (
        f"Question: {row['QuestionText']}\n"
        f"Answer: {row['MC_Answer']}\n"
        f"Is Correct Answer: {'Yes' if row['is_correct'] else 'No'}\n"
        f"Student Explanation: {row['StudentExplanation']}"
    )

train["text"] = train.apply(format_input, axis=1)

# ── Quick-run subset ──────────────────────────────────────────────────────────
if QUICK_RUN:
    train = train.sample(n=min(QUICK_SAMPLES, len(train)), random_state=42).reset_index(drop=True)
    print(f"[QUICK RUN] Using {len(train)} samples")

print(f"Final train shape : {train.shape}")
print("\nSample prompt:")
print(train["text"].iloc[0])

Target classes : 65
[QUICK RUN] Using 256 samples
Final train shape : (256, 11)

Sample prompt:
Question: The probability of an event occurring is \( 0.9 \).

Which of the following most accurately describes the likelihood of the event occurring?
Answer: Likely
Is Correct Answer: Yes
Student Explanation: i think it's b because if it's certain, you're saying that it will happen 100%;however, the number is 0.9 and it isn't "certain" so therefore it would be likely


## 5. Model & Tokenizer Setup

In [15]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=CACHE_DIR,
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Add a padding token if the model lacks one
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

print(f"Vocab size (after pad token) : {len(tokenizer)}")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Vocab size (after pad token) : 151669


In [16]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=n_classes,
    quantization_config=bnb_config,
    device_map="auto",           # H100 single-GPU on Kaggle
    cache_dir=CACHE_DIR,
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Resize embeddings to match tokenizer (needed after adding [PAD])
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded — base dtype : {next(model.parameters()).dtype}")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/442 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-14B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded — base dtype : torch.bfloat16


## 6. LoRA Configuration

In [17]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["score"],   # keep classification head trainable in fp32
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model = model.to(dtype=torch.bfloat16)

model.print_trainable_parameters()

trainable params: 60,625,920 || all params: 14,049,986,560 || trainable%: 0.4315


## 7. Dataset Preparation

In [18]:
df_clean = train[["text", "label"]].copy()
df_clean["label"] = df_clean["label"].astype(np.int64)
df_clean = df_clean.reset_index(drop=True)

train_ds = Dataset.from_pandas(df_clean, preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])

print(f"Tokenized dataset : {train_ds}")

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Tokenized dataset : Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 256
})


## 8. Training

In [19]:
training_args = TrainingArguments(
    output_dir=str(TRAIN_DIR),
    do_train=True,
    do_eval=False,
    save_strategy="no",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    logging_dir=str(LOG_DIR),
    logging_steps=50 if QUICK_RUN else 100,
    remove_unused_columns=False,
    dataloader_drop_last=True,
    dataloader_pin_memory=False,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
print(f"Starting {'quick-run' if QUICK_RUN else 'full'} training — {MODEL_SLUG}")
trainer.train()
print("Training complete.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting quick-run training — Qwen3-14B


Step,Training Loss


Training complete.


## 9. Save Model & Label Encoder

In [21]:
if not QUICK_RUN:
    print(f"Saving model → {SAVE_DIR}")
    trainer.save_model(str(SAVE_DIR))
    tokenizer.save_pretrained(str(SAVE_DIR))

    le_path = SAVE_DIR / "label_encoder.joblib"
    joblib.dump(le, le_path)

    print(f"  Model      : {SAVE_DIR}")
    print(f"  LabelEncoder: {le_path}")
else:
    print("[QUICK RUN] Skipping save — set QUICK_RUN = False for full training + save.")

[QUICK RUN] Skipping save — set QUICK_RUN = False for full training + save.
